In [11]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn import linear_model
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error
from math import sqrt
df = pd.read_excel("../../clean_coast_data.xlsx")

In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25898 entries, 0 to 25897
Data columns (total 32 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   coast_num           25898 non-null  int64         
 1   Area                25898 non-null  object        
 2   Distance from gaza  24095 non-null  float64       
 3   center              25898 non-null  int64         
 4   city rank           25898 non-null  int64         
 5   bathing_season      25898 non-null  int64         
 6   summer_vaction      25898 non-null  int64         
 7   winter              25898 non-null  int64         
 8   river               25898 non-null  int64         
 9   military            25898 non-null  int64         
 10  nature              25898 non-null  int64         
 11  coast_name          25898 non-null  object        
 12  coast_desc          25354 non-null  object        
 13  Date test           25898 non-null  datetime64

In [13]:
df['Area'].unique()

array(['North', 'Center', 'Eilat', 'South'], dtype=object)

In [14]:
# using get_dummies on education column
encoded = pd.get_dummies(df.Area)
# Concatenate the dummies to original dataframe
merged_data = pd.concat([df, encoded], axis=1)
# dropping the original column which was not encoded
merged_data.drop(columns='Area', inplace=True)

In [15]:
merged_data.drop(columns=['coast_name','coast_desc','city','check station','key','coast_num','center'], inplace=True)

Quick & Dirty

In [20]:
merged_data['Date test num'] = merged_data['Date test'].view(np.int64)
without_nulls = merged_data.dropna()
print(df.shape, without_nulls.shape)
without_nulls.sort_values('Date test')
y = without_nulls[['test_result','Date test']]
X = without_nulls.drop('test_result',axis=1)

X_train, X_test, y_train, y_test = train_test_split(X, y, shuffle=True, test_size = 0.2)
X_train_scaler = MinMaxScaler()
X_test_scaler = MinMaxScaler()
X_train_scaled = X_train_scaler.fit_transform(X_train.drop('Date test',axis=1))
X_test_scaled = X_test_scaler.fit_transform(X_test.drop('Date test',axis=1))

mlr = linear_model.LinearRegression()
mlr.fit(X_train_scaled,y_train.test_result)

prediction = mlr.predict(X_train_scaled)
resTrain =  pd.DataFrame(
    data={
        'Prediction':prediction, 
        'Actual': y_train.test_result.values    },
        index=y_train['Date test']
)
prediction = mlr.predict(X_test_scaled)
resTest = pd.DataFrame(
    data={
        'Prediction':prediction,
        'Actual': y_test.test_result.values    },
        index=y_test['Date test']
)

print('rmse train',sqrt(mean_squared_error(resTrain.Prediction, resTrain.Actual)))
print('std train',np.std(resTrain.Actual))
print('Sample')
print(resTrain.sample(10))
print()

print('rmse test',sqrt(mean_squared_error(resTest.Prediction, resTest.Actual)))
print('std test',np.std(resTest.Actual))
print('Sample')
print(resTest.sample(10))
print()

(25898, 32) (21987, 29)
rmse train 0.9605767165233545
std train 0.9891977466950226
Sample
            Prediction  Actual
Date test                     
2011-05-29    3.541016       4
2022-02-28    3.589844       5
2019-05-14    3.802734       5
2006-04-23    3.712891       3
2017-10-17    3.849609       5
2017-01-25    3.671875       2
2015-05-17    3.566406       3
2021-01-31    3.976562       5
2020-01-15    4.023438       2
2006-03-21    3.556641       5

rmse test 6314706645.178715
std test 0.9978671993026803
Sample
              Prediction  Actual
Date test                       
2005-11-01 -6.218825e+09       3
2018-02-15 -5.874388e+09       3
2010-06-06 -6.340783e+09       2
2010-05-16 -6.377873e+09       4
2019-12-15 -5.866188e+09       5
2019-04-15 -5.879509e+09       5
2015-05-17 -6.115212e+09       4
2016-12-27 -5.623066e+09       2
2017-09-24 -6.510190e+09       3
2005-10-18 -6.340290e+09       2



with univariant feature selection

In [31]:
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import f_regression


f_reg_selector = SelectKBest(f_regression, k=17)
f_reg_selector.fit(X, y[['test_result']])
f_reg_support = f_reg_selector.get_support()
f_reg_feature = X.loc[:,f_reg_support].columns.to_list()

uniX = X.loc[:,f_reg_support]

X_train, X_test, y_train, y_test = train_test_split(uniX, y[['test_result']], shuffle=True, test_size = 0.2)
X_train_scaler = MinMaxScaler()
X_test_scaler = MinMaxScaler()
X_train_scaled = X_train_scaler.fit_transform(X_train)
X_test_scaled = X_test_scaler.fit_transform(X_test)

mlr = linear_model.LinearRegression()
mlr.fit(X_train_scaled,y_train.test_result)

prediction = mlr.predict(X_train_scaled)
resTrain =  pd.DataFrame(
    data={
        'Prediction':prediction, 
        'Actual': y_train.test_result.values    },
)
prediction = mlr.predict(X_test_scaled)
resTest = pd.DataFrame(
    data={
        'Prediction':prediction,
        'Actual': y_test.test_result.values    },
)

print('rmse train',sqrt(mean_squared_error(resTrain.Prediction, resTrain.Actual)))
print('std train',np.std(resTrain.Actual))
print('Sample')
print(resTrain.sample(10))
print()

print('rmse test',sqrt(mean_squared_error(resTest.Prediction, resTest.Actual)))
print('std test',np.std(resTest.Actual))
print('Sample')
print(resTest.sample(10))
print()

rmse train 0.9613817477446203
std train 0.9910255491744571
Sample
       Prediction  Actual
3537     3.875000       4
647      3.726562       4
2621     3.691406       4
2912     4.037109       4
3240     3.623047       4
1229     3.699219       4
10171    3.900391       4
1151     3.748047       1
15511    3.628906       2
14255    3.632812       5

rmse test 1861726695.5452805
std test 0.9906075441140143
Sample
        Prediction  Actual
140  -1.188939e+09       3
3668 -2.924830e+08       4
2232 -1.213970e+09       3
2272  4.371796e+08       5
2300 -1.807308e+09       3
3910 -2.095921e+09       5
1328 -1.200382e+09       1
2131 -2.511103e+09       3
1961 -5.485451e+08       5
2630 -8.722107e+07       4



c:\Users\liavc\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\utils\validation.py:72: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  return f(**kwargs)
c:\Users\liavc\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\feature_selection\_univariate_selection.py:302: RuntimeWarning: invalid value encountered in true_divide
  corr /= X_norms
